In [2]:
from datasets import load_dataset

import pandas as pd 

dataset = load_dataset("stanfordnlp/imdb", split="train") 

df = pd.DataFrame(dataset).sample(n= 5000, random_state=42)

df.head() 

,text,label
6868,"Dumb is as dumb does, in this thoroughly unint...",0
24016,I dug out from my garage some old musicals and...,1
9668,After watching this movie I was honestly disap...,0
13640,This movie was nominated for best picture but ...,1
14018,Just like Al Gore shook us up with his painful...,1


In [3]:
print(f"shape: {df.shape}") 

shape: (5000, 2)


In [4]:
print(df["label"].value_counts()) 

label
0    2515
1    2485
Name: count, dtype: int64


In [5]:
df["text_length"]  = df["text"].str.len()

In [6]:
df.head()

,text,label,text_length
6868,"Dumb is as dumb does, in this thoroughly unint...",0,655
24016,I dug out from my garage some old musicals and...,1,796
9668,After watching this movie I was honestly disap...,0,1312
13640,This movie was nominated for best picture but ...,1,775
14018,Just like Al Gore shook us up with his painful...,1,2025


In [7]:
print(df["text_length"].describe())

count    5000.000000
mean     1332.910000
std       995.116408
min        53.000000
25%       698.750000
50%       982.000000
75%      1657.250000
max      6461.000000
Name: text_length, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"] 

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify= y, 
)


print(f"Train:{len(X_train)} | Test: {len(X_test)}")


Train:4000 | Test: 1000


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english") 

X_train_vec = vectorizer.fit_transform(X_train) 

X_test_vec = vectorizer.transform(X_test) 

print(f"Matrix shape:  {X_train_vec.shape}")

Matrix shape:  (4000, 5000)


In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42) 

model.fit(X_train_vec, y_train) 

print("Model Trained") 

Model Trained


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred   = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred) 

print(f"Accuracy : {accuracy:.2%}") 

Accuracy : 85.50%


In [12]:
print(confusion_matrix(y_test, y_pred)) 

[[418  85]
 [ 60 437]]


In [13]:
print(classification_report(y_test,  y_pred, target_names=["negative","positive"]))


              precision    recall  f1-score   support

    negative       0.87      0.83      0.85       503
    positive       0.84      0.88      0.86       497

    accuracy                           0.85      1000
   macro avg       0.86      0.86      0.85      1000
weighted avg       0.86      0.85      0.85      1000



In [ ]:
import numpy as np 

feature_names = np.array(vectorizer.get_feature_names_out())   

coefs = model.coef_[0] 

top_positive = feature_names[np.argsort(coefs)[-15:]]
top_negative = feature_names[np.argsort(coefs)[:15]]


print("Most Positive words: " ,  top_positive) 
print("Most Negative words:  ", top_negative) 

Most Positive words:  ['life' 'fantastic' 'perfect' 'definitely' 'loved' 'amazing' 'classic'
 'enjoyed' 'fun' 'today' 'wonderful' 'love' 'best' 'excellent' 'great']
Most Negative words:   ['worst' 'bad' 'waste' 'terrible' 'awful' 'boring' 'poor' 'worse' 'script'
 'horrible' 'better' 'dull' 'instead' 'just' 'make']


In [15]:
def predict_sentiment(text: str)->str:

    vec = vectorizer.transform([text])

    pred = model.predict(vec)[0] 
    proba  = model.predict_proba(vec)[0].max()


    label = "positvie" if pred == 1 else "negative"

    return f"{label} (confidence : {proba:.0%})" 


print(predict_sentiment("this product exeeded all my expectations,  absolutely fantastic")) 
print(predict_sentiment("Complete waste of money , broke after two days")) 
print(predict_sentiment("It's ok I guess, nothing special")) 
print(predict_sentiment("This is not good at all")) 


positvie (confidence : 73%)
negative (confidence : 77%)
negative (confidence : 64%)
positvie (confidence : 79%)


In [19]:
import joblib

from pathlib import Path 
Path("../models").mkdir(exist_ok = True)

joblib.dump(model,"../models/sentiment_model.joblib")
joblib.dump(vectorizer, "../models/vectorizer.joblib") 

print("saved1 files:", list(Path("../models").glob("*")))

saved1 files: [PosixPath('../models/vectorizer.joblib'), PosixPath('../models/sentiment_model.joblib')]
